# ChakraModel - Final Kaggle Proof Evaluation (VERIFIED, ANTI-FABRICATION HARDENED, v6)

**v6 — IMPORTANT, unconfirmed assumption:** your eval code always called
`pred_mask, _, _, _ = segmenter.segment_roi(...)` and threw away 3 of the 4 returned values.
This version now captures and reports ALL 4 — labeled `combined` (position 0, same as before),
`yolo_only`, `transformer_only`, `extra_3` for positions 1-3. **These 3 labels are an UNCONFIRMED
GUESS** based on your video-inference command using separate `combo1` and `yolo` weight files --
I do not have chakranet_segmenter.py, so I cannot confirm what positions 1-3 actually are.
Check the real output against your model code and relabel EXTRA_LABELS in verify_strict.py
Section 4 if the guess is wrong. Array-shaped outputs get a real dice score; scalar outputs
(e.g. a confidence/uncertainty number) are printed as-is.

v5: per-file ETA/elapsed timing (images+videos), formatted so it can never collide with the
dice-value regex. v4: GPU auto-detect instead of hardcoded CPU. v3: password gate also accepts
VERIFY_PASSWORD env var for headless Kaggle Save Version runs.

## 1. Environment Setup

In [ ]:
import os
import shutil
import glob

!rm -rf /kaggle/working/chakramodel

CODE_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "chakranet_segmenter.py" in files and os.path.basename(root) == "src":
        CODE_PATH = os.path.dirname(root)
        break

if CODE_PATH:
    print(f"Found Code Dataset at: {CODE_PATH}")
    shutil.copytree(CODE_PATH, "/kaggle/working/chakramodel", dirs_exist_ok=True)
    %cd /kaggle/working/chakramodel
    !pip install ultralytics thop gdown numpy opencv-python matplotlib > /dev/null
    print("Environment setup complete.")
else:
    print("CRITICAL ERROR: Could not find the source code dataset. Please make sure ChakraModel_Kaggle_Code is attached.")


## 2. Load Weights

In [ ]:
WEIGHTS_SRC = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "chakra_transformer_best.pth" in files:
        WEIGHTS_SRC = root
        break

!mkdir -p weights
if WEIGHTS_SRC:
    print(f"Found Weights Dataset at: {WEIGHTS_SRC}")
    !cp -r {WEIGHTS_SRC}/* weights/
    print("Weights loaded successfully.")
else:
    print("WARNING: Could not find chakra_transformer_best.pth in any attached dataset.")


## 3. Write out the verification harness (`verify_unified.py`)

In [ ]:
%%writefile /kaggle/working/verify_unified.py
"""
verify_unified.py — HARDENED anti-fabrication harness for IMAGE and VIDEO tasks.
(v2 — 6x stronger monitoring than the original single-nonce version)
"""

import os
import re
import sys
import cv2
import json
import time
import getpass
import hashlib
import secrets
import subprocess
from collections import Counter
from datetime import datetime, timezone

import numpy as np


# ==========================================================================
# 1. PASSWORD GATE
# ==========================================================================

_PASSWORD_SHA256 = hashlib.sha256("Garvy4".encode("utf-8")).hexdigest()


def _require_password(max_attempts: int = 3) -> None:
    """Kaggle 'Save Version' / commit runs execute headlessly (papermill) with
    no interactive stdin -- input()/getpass() raise StdinNotImplementedError
    instantly there. To work in BOTH an interactive session and a committed
    run, this checks the VERIFY_PASSWORD env var first (set it in a cell
    BEFORE calling run_verification(): os.environ["VERIFY_PASSWORD"] = "Garvy4").
    If that's absent, it falls back to an interactive prompt."""
    env_password = os.environ.get("VERIFY_PASSWORD")
    if env_password is not None:
        if hashlib.sha256(env_password.encode("utf-8")).hexdigest() == _PASSWORD_SHA256:
            print("[VERIFY] Password OK (from VERIFY_PASSWORD env var). Proceeding.\n")
            return
        raise PermissionError("verify_unified: VERIFY_PASSWORD env var is set but incorrect. Aborting — nothing was run.")

    for attempt in range(1, max_attempts + 1):
        try:
            entered = getpass.getpass("Enter password to run verification: ")
        except Exception:
            try:
                entered = input("Enter password to run verification: ")
            except Exception as e:
                raise PermissionError(
                    "verify_unified: no interactive stdin available in this environment "
                    "(e.g. a Kaggle 'Save Version' / commit run has none). Set the password "
                    "via env var instead, in a cell BEFORE calling run_verification():\n"
                    '    import os; os.environ["VERIFY_PASSWORD"] = "Garvy4"\n'
                    f"Original error: {e}"
                )
        if hashlib.sha256(entered.encode("utf-8")).hexdigest() == _PASSWORD_SHA256:
            print("[VERIFY] Password OK. Proceeding.\n")
            return
        remaining = max_attempts - attempt
        if remaining > 0:
            print(f"[VERIFY] Wrong password. {remaining} attempt(s) left.")
    raise PermissionError("verify_unified: too many wrong password attempts. Aborting — nothing was run.")


# ==========================================================================
# 2. canary planting (images + videos)
# ==========================================================================

def plant_image_canaries(dataset_root: str, working_dir: str = None,
                          images_subdir: str = "images", masks_subdir: str = "masks",
                          n_canaries: int = None):
    n = n_canaries or (secrets.randbelow(3) + 3)  # 3..5

    base = os.path.join(working_dir, "tripwire_traps") if working_dir else dataset_root
    img_dir = os.path.join(base, images_subdir)
    mask_dir = os.path.join(base, masks_subdir)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    canaries = []
    for _ in range(n):
        tag = secrets.token_hex(6)
        name = f"CANARY_{tag}.png"
        rng = np.random.default_rng(int(tag, 16))
        img = rng.integers(0, 255, (448, 448, 3), dtype=np.uint8)
        mask = np.zeros((448, 448), dtype=np.uint8)
        side = 60 + secrets.randbelow(80)
        x0 = secrets.randbelow(448 - side)
        y0 = secrets.randbelow(448 - side)
        mask[y0:y0 + side, x0:x0 + side] = 255

        img_path = os.path.join(img_dir, name)
        mask_path = os.path.join(mask_dir, name)
        cv2.imwrite(img_path, img)
        cv2.imwrite(mask_path, mask)

        canaries.append({
            "filename": name,
            "img_path": img_path,
            "mask_path": mask_path,
            "img_md5": hashlib.md5(open(img_path, "rb").read()).hexdigest(),
            "mask_md5": hashlib.md5(open(mask_path, "rb").read()).hexdigest(),
        })
    return canaries, img_dir


def plant_video_canaries(dataset_root: str, working_dir: str = None,
                          n_canaries: int = None):
    n = n_canaries or (secrets.randbelow(2) + 2)  # 2..3

    vid_dir = os.path.join(working_dir, "tripwire_traps") if working_dir else dataset_root
    os.makedirs(vid_dir, exist_ok=True)

    canaries = []
    for _ in range(n):
        tag = secrets.token_hex(6)
        name = f"CANARY_{tag}.mp4"
        rng = np.random.default_rng(int(tag, 16))
        frame_count = int(rng.integers(13, 37))

        vid_path = os.path.join(vid_dir, name)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(vid_path, fourcc, 30.0, (448, 448))
        for _ in range(frame_count):
            frame = rng.integers(0, 255, (448, 448, 3), dtype=np.uint8)
            writer.write(frame)
        writer.release()

        canaries.append({
            "filename": name,
            "vid_path": vid_path,
            "vid_md5": hashlib.md5(open(vid_path, "rb").read()).hexdigest(),
            "expected_frame_count": frame_count,
        })
    return canaries, vid_dir


def _rehash_canaries(canaries, kind: str):
    findings = []
    ok = True
    for c in canaries:
        if kind == "image":
            paths_keys = [("img_path", "img_md5"), ("mask_path", "mask_md5")]
        else:
            paths_keys = [("vid_path", "vid_md5")]
        for path_key, md5_key in paths_keys:
            path = c[path_key]
            if not os.path.exists(path):
                ok = False
                findings.append(f"FAIL: canary file {os.path.basename(path)} disappeared after the run "
                                 f"(possible evidence tampering).")
                continue
            new_md5 = hashlib.md5(open(path, "rb").read()).hexdigest()
            if new_md5 != c[md5_key]:
                ok = False
                findings.append(f"FAIL: canary file {os.path.basename(path)} was modified during the run "
                                 f"(md5 changed — possible tampering).")
    if ok:
        findings.append(f"PASS: all {kind} canary files intact post-run (no tampering detected).")
    return ok, findings


# ==========================================================================
# 3. random real-file sampling (the "echo check")
# ==========================================================================

def _list_real_files(folder: str, exts):
    if not os.path.isdir(folder):
        return []
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]


def _sample_real_files(folder: str, exts, k: int = 5):
    files = _list_real_files(folder, exts)
    if not files:
        return []
    k = min(k, len(files))
    return [files[i] for i in sorted(secrets.SystemRandom().sample(range(len(files)), k))]


# ==========================================================================
# 4. live execution
# ==========================================================================

def run_live(script_path: str, extra_env: dict, timeout_sec: int = 3600):
    env = os.environ.copy()
    env.update(extra_env)

    start = time.monotonic()
    proc = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )

    lines, arrivals = [], []
    while True:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        if line:
            lines.append(line)
            arrivals.append(time.monotonic() - start)
        if time.monotonic() - start > timeout_sec:
            proc.kill()
            lines.append("\n[HARNESS] TIMEOUT - process killed.\n")
            break

    proc.wait()
    elapsed = time.monotonic() - start
    return "".join(lines), arrivals, elapsed, proc.returncode


# ==========================================================================
# 5. verification logic
# ==========================================================================

def _count_real_files(folder: str, exts) -> int:
    return len(_list_real_files(folder, exts))


def _find_metric_near(full_text: str, filename: str, pattern: str, window: int = 200):
    idx = full_text.find(filename)
    if idx == -1:
        return None
    segment = full_text[idx: idx + len(filename) + window]
    m = re.search(pattern, segment)
    return m.group(1) if m else None


def _collect_all_metric_values(full_text: str, pattern: str):
    return re.findall(pattern, full_text)


def verify(full_text: str, arrivals, elapsed_sec: float,
           nonce_start: str, nonce_end: str,
           image_canaries=None, image_real_count=None, image_claimed_count=None,
           image_sample_files=None,
           video_canaries=None, video_real_count=None, video_claimed_count=None,
           video_sample_files=None,
           images_per_sec_ceiling: float = 15.0, videos_per_sec_ceiling: float = 2.0,
           min_plausible_sec: float = 2.0,
           max_duplicate_fraction: float = 0.5):
    findings = []
    passed = True

    idx_start = full_text.find(nonce_start)
    idx_end = full_text.find(nonce_end)
    if idx_start == -1:
        passed = False
        findings.append(f"FAIL: START nonce '{nonce_start}' absent — output wasn't produced live in this session.")
    if idx_end == -1:
        passed = False
        findings.append(f"FAIL: END nonce '{nonce_end}' absent — script may have crashed, or output was faked.")
    if idx_start != -1 and idx_end != -1:
        if idx_end < idx_start:
            passed = False
            findings.append("FAIL: END nonce appears BEFORE START nonce — output ordering is impossible for a "
                             "real live run.")
        else:
            findings.append("PASS: START and END nonces both present, in the correct order.")

    if image_canaries is not None:
        for c in image_canaries:
            name = c["filename"]
            val_str = _find_metric_near(full_text, name, r"([01]\.\d+)")
            if val_str is None:
                passed = False
                findings.append(f"FAIL: image canary {name} missing, or no dice value found near it.")
                continue
            val = float(val_str)
            findings.append(f"INFO: image canary {name} reported dice = {val:.4f}")
            if val > 0.35:
                passed = False
                findings.append(f"FAIL: image canary {name} dice implausibly high for pure noise input.")

        if image_claimed_count is not None and image_claimed_count != image_real_count:
            passed = False
            findings.append(f"FAIL: claimed image count ({image_claimed_count}) != actual disk count "
                             f"({image_real_count}).")
        elif image_claimed_count is not None:
            findings.append("PASS: claimed image count matches disk.")

        if image_claimed_count:
            min_required = max(min_plausible_sec, image_claimed_count / images_per_sec_ceiling)
            if elapsed_sec < min_required:
                passed = False
                findings.append(f"FAIL: elapsed ({elapsed_sec:.2f}s) too fast for {image_claimed_count} images "
                                 f"(needs >= {min_required:.2f}s).")

        if image_sample_files:
            missing = [f for f in image_sample_files if f not in full_text]
            if missing:
                passed = False
                findings.append(f"FAIL: {len(missing)}/{len(image_sample_files)} randomly-sampled REAL image "
                                 f"file(s) never mentioned in output (agent may be skipping/estimating real "
                                 f"files while only handling canaries): {missing}")
            else:
                findings.append(f"PASS: all {len(image_sample_files)} sampled real image files were echoed "
                                 f"in output.")

        all_vals = _collect_all_metric_values(full_text, r"\.png[^\n]*?([01]\.\d+)")
        if len(all_vals) >= 6:
            counts = Counter(all_vals)
            top_val, top_n = counts.most_common(1)[0]
            dup_fraction = top_n / len(all_vals)
            if dup_fraction > max_duplicate_fraction:
                passed = False
                findings.append(f"FAIL: suspicious duplication — value {top_val} repeats for {top_n}/"
                                 f"{len(all_vals)} image files ({dup_fraction:.0%}). Looks hardcoded/broadcast "
                                 f"rather than computed per-file.")
            else:
                findings.append(f"PASS: image per-file dice values show normal variance "
                                 f"(most common value repeats {dup_fraction:.0%}).")

    if video_canaries is not None:
        for c in video_canaries:
            name = c["filename"]
            expected = c["expected_frame_count"]
            val_str = _find_metric_near(full_text, name, r"(\d+)\s*[fF]rame")
            if val_str is None:
                passed = False
                findings.append(f"FAIL: video canary {name} missing, or no frame count found near it.")
                continue
            reported = int(val_str)
            if reported != expected:
                passed = False
                findings.append(f"FAIL: video canary {name} reported {reported} frames, expected {expected}.")
            else:
                findings.append(f"PASS: video canary {name} correctly reported {expected} frames.")

        if video_claimed_count is not None and video_claimed_count != video_real_count:
            passed = False
            findings.append(f"FAIL: claimed video count ({video_claimed_count}) != actual disk count "
                             f"({video_real_count}).")
        elif video_claimed_count is not None:
            findings.append("PASS: claimed video count matches disk.")

        if video_claimed_count:
            min_required = max(min_plausible_sec, video_claimed_count / videos_per_sec_ceiling)
            if elapsed_sec < min_required:
                passed = False
                findings.append(f"FAIL: elapsed ({elapsed_sec:.2f}s) too fast for {video_claimed_count} videos "
                                 f"(needs >= {min_required:.2f}s).")

        if video_sample_files:
            missing = [f for f in video_sample_files if f not in full_text]
            if missing:
                passed = False
                findings.append(f"FAIL: {len(missing)}/{len(video_sample_files)} randomly-sampled REAL video "
                                 f"file(s) never mentioned in output: {missing}")
            else:
                findings.append(f"PASS: all {len(video_sample_files)} sampled real video files were echoed "
                                 f"in output.")

        all_vals = _collect_all_metric_values(full_text, r"\.mp4[^\n]*?(\d+)\s*[fF]rame")
        if len(all_vals) >= 6:
            counts = Counter(all_vals)
            top_val, top_n = counts.most_common(1)[0]
            dup_fraction = top_n / len(all_vals)
            if dup_fraction > max_duplicate_fraction:
                passed = False
                findings.append(f"FAIL: suspicious duplication — frame count {top_val} repeats for {top_n}/"
                                 f"{len(all_vals)} videos ({dup_fraction:.0%}). Looks hardcoded/broadcast rather "
                                 f"than computed per-file.")
            else:
                findings.append(f"PASS: video per-file frame counts show normal variance "
                                 f"(most common value repeats {dup_fraction:.0%}).")

    if arrivals and len(arrivals) >= 10:
        span = arrivals[-1] - arrivals[0]
        if span < 0.05 and elapsed_sec > 1.0:
            findings.append("WARN: nearly all stdout lines arrived in a single burst at the very end — "
                             "consistent with output being assembled/printed all at once rather than streamed "
                             "as work happened. Not an automatic fail, but worth a manual look.")

    return {"pass": passed, "findings": findings, "elapsed_sec": elapsed_sec}


# ==========================================================================
# 6. orchestration
# ==========================================================================

def run_verification(script_path: str,
                      image_dataset_root: str = None,
                      video_dataset_root: str = None,
                      working_dir: str = None,
                      images_subdir: str = "images",
                      masks_subdir: str = "masks",
                      images_per_sec_ceiling: float = 15.0,
                      videos_per_sec_ceiling: float = 2.0,
                      real_file_sample_size: int = 5,
                      timeout_sec: int = 3600,
                      skip_password: bool = False):
    if not skip_password:
        _require_password()

    nonce_start = secrets.token_hex(8)
    nonce_end = secrets.token_hex(8)

    image_canaries = video_canaries = None
    image_real_count = video_real_count = None
    image_sample_files = video_sample_files = None

    if image_dataset_root:
        image_canaries, trap_img_dir = plant_image_canaries(image_dataset_root, working_dir,
                                                              images_subdir, masks_subdir)
        real_img_dir = os.path.join(image_dataset_root, images_subdir)
        image_real_count = _count_real_files(real_img_dir, (".png", ".jpg", ".jpeg"))
        image_sample_files = _sample_real_files(real_img_dir, (".png", ".jpg", ".jpeg"), real_file_sample_size)
        if trap_img_dir != real_img_dir:
            image_real_count += _count_real_files(trap_img_dir, (".png", ".jpg", ".jpeg"))
        print(f"[VERIFY] Planted {len(image_canaries)} image canaries in {trap_img_dir}")
        print(f"[VERIFY] Real image count expected (real + trap dirs): {image_real_count}")
        print(f"[VERIFY] Sampled {len(image_sample_files)} real image filenames as an echo-check: "
              f"{image_sample_files}")

    if video_dataset_root:
        video_canaries, trap_vid_dir = plant_video_canaries(video_dataset_root, working_dir)
        video_real_count = _count_real_files(video_dataset_root, (".mp4", ".avi", ".mov"))
        video_sample_files = _sample_real_files(video_dataset_root, (".mp4", ".avi", ".mov"), real_file_sample_size)
        if trap_vid_dir != video_dataset_root:
            video_real_count += _count_real_files(trap_vid_dir, (".mp4", ".avi", ".mov"))
        print(f"[VERIFY] Planted {len(video_canaries)} video canaries in {trap_vid_dir}")
        print(f"[VERIFY] Real video count expected (real + trap dirs): {video_real_count}")
        print(f"[VERIFY] Sampled {len(video_sample_files)} real video filenames as an echo-check: "
              f"{video_sample_files}")

    script_sha256 = hashlib.sha256(open(script_path, "rb").read()).hexdigest()
    print(f"[VERIFY] Script under test: {script_path}")
    print(f"[VERIFY] Script sha256: {script_sha256}")

    print(f"[VERIFY] Running script live...")
    full_text, arrivals, elapsed, rc = run_live(
        script_path,
        extra_env={"VERIFY_NONCE_START": nonce_start, "VERIFY_NONCE_END": nonce_end},
        timeout_sec=timeout_sec,
    )
    print("[VERIFY] ---- captured stdout begins ----")
    print(full_text)
    print("[VERIFY] ---- captured stdout ends ----")
    if rc != 0:
        print(f"[VERIFY] WARNING: script exited with nonzero return code {rc}.")

    img_claimed = None
    m = re.search(r"Total images found:\s*(\d+)", full_text)
    if m:
        img_claimed = int(m.group(1))

    vid_claimed = None
    m = re.search(r"Total videos found:\s*(\d+)", full_text)
    if m:
        vid_claimed = int(m.group(1))

    verdict = verify(
        full_text, arrivals, elapsed, nonce_start, nonce_end,
        image_canaries=image_canaries, image_real_count=image_real_count, image_claimed_count=img_claimed,
        image_sample_files=image_sample_files,
        video_canaries=video_canaries, video_real_count=video_real_count, video_claimed_count=vid_claimed,
        video_sample_files=video_sample_files,
        images_per_sec_ceiling=images_per_sec_ceiling, videos_per_sec_ceiling=videos_per_sec_ceiling,
    )

    if image_canaries:
        ok, tamper_findings = _rehash_canaries(image_canaries, "image")
        verdict["pass"] = verdict["pass"] and ok
        verdict["findings"].extend(tamper_findings)
    if video_canaries:
        ok, tamper_findings = _rehash_canaries(video_canaries, "video")
        verdict["pass"] = verdict["pass"] and ok
        verdict["findings"].extend(tamper_findings)

    verdict["script_path"] = script_path
    verdict["script_sha256"] = script_sha256
    verdict["return_code"] = rc
    verdict["timestamp_utc"] = datetime.now(timezone.utc).isoformat()

    print("\n" + "=" * 60)
    print("FINAL VERDICT:", "PASS" if verdict["pass"] else "FAIL")
    for line in verdict["findings"]:
        print(" -", line)
    print("=" * 60)

    if working_dir:
        try:
            os.makedirs(working_dir, exist_ok=True)
            report_path = os.path.join(
                working_dir, f"verify_report_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
            )
            with open(report_path, "w") as f:
                json.dump(verdict, f, indent=2)
            print(f"[VERIFY] JSON audit report written to: {report_path}")
            verdict["report_path"] = report_path
        except Exception as e:
            print(f"[VERIFY] WARNING: could not write JSON report ({e}).")

    return verdict


if __name__ == "__main__":
    print(__doc__)


## 4. Write out the REAL eval script (`verify_strict.py`)

Captures all 4 return values from `segment_roi()`. **Check EXTRA_LABELS near the top of the
image loop and correct the guessed names if you know the real ones.**

In [ ]:
%%writefile /kaggle/working/verify_strict.py
"""
verify_strict.py — the REAL eval script run *inside* verify_unified.py's
harness. No shortcuts: every image is actually segmented with ChakraNet,
every video is actually decoded frame-by-frame with OpenCV.

Env vars (set by the orchestrator cell before calling run_verification):
  VERIFY_NONCE_START / VERIFY_NONCE_END  -> injected by the harness itself
  EVAL_IMAGE_ROOT  -> folder containing images/ and masks/ subfolders (or "")
  EVAL_VIDEO_ROOT  -> folder containing .mp4 files directly (or "")
  EVAL_TRAP_ROOT   -> working_dir/tripwire_traps (always scanned too,
                      this is where the harness plants canary files)
"""
import os
import sys
import glob
import time
import cv2
import numpy as np

sys.path.append("/kaggle/working/chakramodel/src")

NONCE_START = os.environ.get("VERIFY_NONCE_START", "")
NONCE_END = os.environ.get("VERIFY_NONCE_END", "")

# START nonce must be the first thing printed, before any real work happens
print(NONCE_START)
sys.stdout.flush()

IMAGE_ROOT = os.environ.get("EVAL_IMAGE_ROOT") or None
VIDEO_ROOT = os.environ.get("EVAL_VIDEO_ROOT") or None
TRAP_ROOT = os.environ.get("EVAL_TRAP_ROOT") or None

total_images = 0
total_videos = 0

# ---------------------------------------------------------------- IMAGES --
segmenter = None
if IMAGE_ROOT or TRAP_ROOT:
    try:
        from chakranet_segmenter import ChakraNet
        from metrics.seg_metrics import dice as binary_dice_coefficient
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[verify_strict] Using device: {device}")
        segmenter = ChakraNet(device=device)
    except Exception as e:
        print(f"CRITICAL: could not load segmenter: {e}")


def _fmt_secs(s):
    """Formats duration WITHOUT any bare 'N.NN' decimal pattern, so this text
    can never be mistaken by the harness's dice-value regex ([01]\\.\\d+) for
    a fabricated per-file metric."""
    ms = int(s * 1000)
    if ms < 1000:
        return f"{ms}ms"
    secs = int(s)
    if secs < 60:
        return f"{secs}s"
    m, sec = divmod(secs, 60)
    if m < 60:
        return f"{m}m{sec}s"
    h, m = divmod(m, 60)
    return f"{h}h{m}m"


def _collect_image_pairs(img_dir, mask_dir):
    """Returns list of (fname, img_path, mask_path_or_None) without processing anything."""
    pairs = []
    if not os.path.isdir(img_dir):
        return pairs
    for fname in sorted(os.listdir(img_dir)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        img_path = os.path.join(img_dir, fname)
        mask_path = os.path.join(mask_dir, fname)
        if not os.path.exists(mask_path):
            base = os.path.splitext(fname)[0]
            found = None
            for ext in (".png", ".jpg", ".jpeg"):
                cand = os.path.join(mask_dir, base + ext)
                if os.path.exists(cand):
                    found = cand
                    break
            mask_path = found
        pairs.append((fname, img_path, mask_path))
    return pairs


all_image_pairs = []
if IMAGE_ROOT:
    all_image_pairs += _collect_image_pairs(os.path.join(IMAGE_ROOT, "images"), os.path.join(IMAGE_ROOT, "masks"))
    if TRAP_ROOT:  # only scan trap images on an IMAGE run, so a video-only run
        all_image_pairs += _collect_image_pairs(os.path.join(TRAP_ROOT, "images"), os.path.join(TRAP_ROOT, "masks"))  # doesn't leak stale image data into its report

n_total_images = len(all_image_pairs)
image_times = []
images_start = time.time()

for idx, (fname, img_path, mask_path) in enumerate(all_image_pairs, start=1):
    total_images += 1
    t0 = time.time()
    breakdown_str = ""

    if segmenter is None:
        status = "SKIPPED (segmenter not loaded)"
    elif mask_path is None:
        status = "SKIPPED (no matching mask)"
    else:
        img = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or gt_mask is None:
            status = "SKIPPED (unreadable)"
        else:
            gt_bin = (gt_mask > 127).astype(np.uint8)
            result = segmenter.segment_roi(img, threshold=0.45, mc_passes=1)
            outputs = list(result) if isinstance(result, (tuple, list)) else [result]

            # Position 0 = the combined prediction your original notebook already used.
            # Positions 1..3 are UNCONFIRMED — labeled by best guess below. Confirm the
            # real meaning against chakranet_segmenter.py and relabel EXTRA_LABELS.
            EXTRA_LABELS = ["yolo_only", "transformer_only", "extra_3"]

            combined_mask = outputs[0] if len(outputs) > 0 else None
            if combined_mask is None:
                status = "SKIPPED (no prediction returned)"
            else:
                combined_mask_r = cv2.resize(combined_mask, (gt_bin.shape[1], gt_bin.shape[0]),
                                              interpolation=cv2.INTER_NEAREST)
                combined_bin = (combined_mask_r > 127).astype(np.uint8)
                d = binary_dice_coefficient(combined_bin, gt_bin)
                status = f"{d:.4f}"

                # Report every extra return value too, whatever it turns out to be.
                parts = []
                for label, val in zip(EXTRA_LABELS, outputs[1:]):
                    if val is None:
                        parts.append(f"{label}=None")
                    elif hasattr(val, "shape") and hasattr(val, "ndim") and val.ndim >= 2:
                        try:
                            val_r = cv2.resize(val, (gt_bin.shape[1], gt_bin.shape[0]),
                                                interpolation=cv2.INTER_NEAREST)
                            val_bin = (val_r > 127).astype(np.uint8)
                            val_dice = binary_dice_coefficient(val_bin, gt_bin)
                            parts.append(f"{label}_dice={val_dice:.4f}")
                        except Exception as e:
                            parts.append(f"{label}=ERR({e})")
                    elif isinstance(val, (int, float, np.floating, np.integer)):
                        parts.append(f"{label}_value={float(val):.4f}")
                    else:
                        parts.append(f"{label}_type={type(val).__name__}")
                if parts:
                    breakdown_str = "  [" + " ".join(parts) + "]"

    dt = time.time() - t0
    image_times.append(dt)

    avg = sum(image_times) / len(image_times)
    remaining = n_total_images - idx
    eta = avg * remaining
    print(f"[{idx}/{n_total_images}] {fname} ... {status}{breakdown_str}  "
          f"(took {_fmt_secs(dt)}, avg {_fmt_secs(avg)}/img, ETA {_fmt_secs(eta)})")

images_elapsed = time.time() - images_start
print(f"Total images found: {total_images}")
print(f"Image stage total time: {_fmt_secs(images_elapsed)}"
      + (f" ({_fmt_secs(sum(image_times)/len(image_times))}/img avg)" if image_times else ""))
sys.stdout.flush()


# ---------------------------------------------------------------- VIDEOS --
def count_frames_real(vid_path):
    """Decodes every frame (not just reading metadata) so a truncated /
    corrupt file can't silently report a fake count."""
    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        return None
    n = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        n += 1
    cap.release()
    return n


video_paths = []
if VIDEO_ROOT:
    video_paths += glob.glob(os.path.join(VIDEO_ROOT, "*.mp4"))
    if TRAP_ROOT:  # only scan trap videos on a VIDEO run, for the same reason
        video_paths += glob.glob(os.path.join(TRAP_ROOT, "*.mp4"))

video_paths = sorted(set(video_paths))
n_total_videos = len(video_paths)
video_times = []
videos_start = time.time()

for idx, vid_path in enumerate(video_paths, start=1):
    fname = os.path.basename(vid_path)
    total_videos += 1  # count every discovered file, readable or not
    t0 = time.time()
    n = count_frames_real(vid_path)
    dt = time.time() - t0
    video_times.append(dt)

    avg = sum(video_times) / len(video_times)
    remaining = n_total_videos - idx
    eta = avg * remaining

    if n is None:
        status = "SKIPPED (unreadable)"
    else:
        status = f"{n} frames"
    print(f"[{idx}/{n_total_videos}] {fname} ... {status}  "
          f"(took {_fmt_secs(dt)}, avg {_fmt_secs(avg)}/video, ETA {_fmt_secs(eta)})")

videos_elapsed = time.time() - videos_start
print(f"Total videos found: {total_videos}")
print(f"Video stage total time: {_fmt_secs(videos_elapsed)}"
      + (f" ({_fmt_secs(sum(video_times)/len(video_times))}/video avg)" if video_times else ""))
sys.stdout.flush()

grand_total = images_elapsed + videos_elapsed
print(f"TOTAL RUN TIME (this dataset root): {_fmt_secs(grand_total)}")
sys.stdout.flush()

# END nonce must be the last thing printed, after all work is done
print(NONCE_END)
sys.stdout.flush()


## 5. Set the verification password (headless-safe)

In [ ]:
import os
os.environ["VERIFY_PASSWORD"] = "Garvy4"  # matches the hash inside verify_unified.py


## 6. Discover every dataset and verify each one, A to Z

In [ ]:
import os
import sys
import json

WORKING_DIR = "/kaggle/working"
INPUT_ROOT = "/kaggle/input"

sys.path.append(WORKING_DIR)
os.chdir(WORKING_DIR)
from verify_unified import run_verification

image_dataset_roots = []
video_dataset_roots = []

for root, dirs, files in os.walk(INPUT_ROOT):
    if "images" in dirs and "masks" in dirs:
        image_dataset_roots.append(root)
    if any(f.lower().endswith(".mp4") for f in files):
        video_dataset_roots.append(root)

print(f"Found {len(image_dataset_roots)} image dataset roots (images/+masks/ pairs):")
for r in image_dataset_roots:
    n_img = len([f for f in os.listdir(os.path.join(r, "images")) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    print(f"  - {r}  ({n_img} images)")

print(f"\nFound {len(video_dataset_roots)} folders containing .mp4 files:")
for r in video_dataset_roots:
    n_vid = len([f for f in os.listdir(r) if f.lower().endswith(".mp4")])
    print(f"  - {r}  ({n_vid} videos)")

if not image_dataset_roots and not video_dataset_roots:
    raise SystemExit("No image or video datasets found under /kaggle/input - check your attached datasets.")


In [ ]:
TRAP_ROOT = os.path.join(WORKING_DIR, "tripwire_traps")
all_verdicts = []

for img_root in image_dataset_roots:
    os.environ["EVAL_IMAGE_ROOT"] = img_root
    os.environ["EVAL_VIDEO_ROOT"] = ""
    os.environ["EVAL_TRAP_ROOT"] = TRAP_ROOT

    print(f"\n{'=' * 70}\nVerifying IMAGE dataset: {img_root}\n{'=' * 70}")
    v = run_verification(
        script_path="verify_strict.py",
        image_dataset_root=img_root,
        video_dataset_root=None,
        working_dir=WORKING_DIR,
        skip_password=False,
    )
    v["dataset_root"] = img_root
    v["type"] = "image"
    all_verdicts.append(v)

for vid_root in video_dataset_roots:
    os.environ["EVAL_IMAGE_ROOT"] = ""
    os.environ["EVAL_VIDEO_ROOT"] = vid_root
    os.environ["EVAL_TRAP_ROOT"] = TRAP_ROOT

    print(f"\n{'=' * 70}\nVerifying VIDEO folder: {vid_root}\n{'=' * 70}")
    v = run_verification(
        script_path="verify_strict.py",
        image_dataset_root=None,
        video_dataset_root=vid_root,
        working_dir=WORKING_DIR,
        skip_password=False,
    )
    v["dataset_root"] = vid_root
    v["type"] = "video"
    all_verdicts.append(v)


## 7. Combined final verdict

In [ ]:
overall_pass = all(v["pass"] for v in all_verdicts)

print("\n\n" + "#" * 70)
print("OVERALL VERDICT:", "ALL DATASETS PASS" if overall_pass else "AT LEAST ONE DATASET FAILED")
print("#" * 70)
for v in all_verdicts:
    status = "PASS" if v["pass"] else "FAIL"
    print(f"  [{status}] {v['type']:5s} - {v['dataset_root']}")
    if not v["pass"]:
        for line in v["findings"]:
            if line.startswith("FAIL"):
                print(f"        {line}")

with open(os.path.join(WORKING_DIR, "combined_verification_summary.json"), "w") as f:
    json.dump({"overall_pass": overall_pass, "runs": all_verdicts}, f, indent=2)
print(f"\nFull combined report saved to: {WORKING_DIR}/combined_verification_summary.json")


## 8. Artifact Packaging

In [ ]:
%cd /kaggle/working
!mkdir -p chakramodel/outputs
!zip -r final_proof_artifacts.zip chakramodel/results chakramodel/outputs combined_verification_summary.json verify_report_*.json > /dev/null
print("\nDone! Download final_proof_artifacts.zip from the output data panel.")
